# 1 · Your first simulation

What a market looks like, and what you just ran.

By the end you will have built a universe, run it forward, read the order
book, and, the part that matters, checked that the whole thing reproduces
exactly.

**Prerequisites:** `pip install tradefloor`. Nothing else; the core has no
dependencies.

In [1]:
import struct
import tradefloor as tf

print("tradefloor", tf.version())
print("shipped model preset:", tf.model_preset()["name"])

tradefloor 0.6.0
shipped model preset: pt-v16


## Building a universe

`Universe.random` generates a plausible cross-section: market caps spread
across all four spread tiers, P/E ratios scattered around each sector's
anchor, and about one name in nine a loss-maker, so the book-value valuation
path gets exercised.

There are two independent seeds. The universe seed picks the companies; the
simulation seed picks the market's draws. Keeping them separate lets you
hold the companies fixed and vary the market, which is how you estimate
variance across runs.

Roster order matters. The engine iterates instruments in index order and
draws as it goes, so sorting a universe gives you a different market from
the same seed.

In [2]:
universe = tf.Universe.random(20, seed=111)

print(f"{len(universe)} instruments\n")
print(f"{'ticker':8s} {'sector':24s} {'price':>9s} {'beta':>6s} {'avg volume':>12s}")
for inst in list(universe)[:6]:
    print(f"{inst.ticker:8s} {inst.sector:24s} {inst.initial_price:9.2f} "
          f"{inst.beta:6.2f} {inst.avg_volume:12,.0f}")

20 instruments

ticker   sector                       price   beta   avg volume
AAA      technology                  200.45   0.93       75,955
AAB      financial_services           11.56   0.85    5,360,256
AAC      healthcare                   23.55   0.64    2,544,755
AAD      energy                        6.61   1.12    5,656,606
AAE      consumer_discretionary       18.79   0.98  373,407,166
AAF      consumer_staples              4.33   0.47  652,341,695


## Running the market

`Engine` takes the universe and a seed. `run_days` advances whole trading
days: open, a session of ticks, then the close, which is where the daily
GARCH update and the macro chain step happen.

`ticks_per_day` defaults to 390, one per minute of a US trading session.
Lowering it makes runs faster and coarser; it changes the market, so it is
part of what you cite.

In [3]:
engine = tf.Engine(universe=universe, seed=7)
engine.run_days(10)

raw = engine.prices()
prices = struct.unpack(f"<{len(raw) // 8}d", raw)

print(f"{'ticker':8s} {'start':>9s} {'day 10':>9s} {'change':>9s}")
for inst, now in list(zip(universe, prices))[:6]:
    pct = (now / inst.initial_price - 1) * 100
    print(f"{inst.ticker:8s} {inst.initial_price:9.2f} {now:9.2f} {pct:8.2f}%")

ticker       start    day 10    change
AAA         200.45    193.71    -3.36%
AAB          11.56     11.50    -0.53%
AAC          23.55     22.02    -6.50%
AAD           6.61      6.22    -5.83%
AAE          18.79     22.07    17.48%
AAF           4.33      4.47     3.30%


`prices()` hands back raw little-endian `f64` bytes rather than a list. That
is deliberate: `numpy.frombuffer(raw, dtype="<f8")` adopts them without
copying, which matters when you are pulling prices every tick in a training
loop. `struct.unpack` above avoids the numpy dependency for this notebook.

## The order book

An actual book with price levels and price-time priority, rather than a
spread model. It is what makes size cost money.

In [4]:
ticker = engine.tickers[0]
book = engine.book(ticker)

print(f"{ticker}")
print(f"  best bid   {book.best_bid:9.2f}")
print(f"  best ask   {book.best_ask:9.2f}")
print(f"  mid        {book.mid_price:9.2f}")
print(f"  spread     {book.best_ask - book.best_bid:9.4f}"
      f"   ({(book.best_ask - book.best_bid) / book.mid_price * 1e4:.1f} bps)")
print(f"  bid depth  {book.depth('buy'):9,.0f} shares")
print(f"  ask depth  {book.depth('sell'):9,.0f} shares")

AAA
  best bid      193.64
  best ask      193.85
  mid           193.75
  spread        0.2100   (10.8 bps)
  bid depth      3,806 shares
  ask depth      2,635 shares


## The economy

VIX, market sentiment, inflation, the policy rate and credit spreads all
evolve, with a five-phase business cycle running underneath them. They run on
different clocks, though: VIX and sentiment are redrawn every trading day,
inflation lands on a monthly print, and the policy rate waits for a
rate-setting meeting. That is why, ten days in, the VIX and fear/greed
figures below have moved off their opening values while the federal funds
rate and inflation still read theirs exactly.

Conditions move while your strategy is trading. Run it long enough and every
one of them moves.

In [5]:
m = engine.macro_state
print(f"  VIX                  {m.vix:8.2f}")
print(f"  federal funds rate   {m.federal_funds_rate:8.4f}   (a fraction, not percent)")
print(f"  inflation            {m.inflation_rate:8.4f}")
print(f"  fear/greed           {m.fear_greed_index:8.1f}")

  VIX                     12.70
  federal funds rate     0.0250   (a fraction, not percent)
  inflation              0.0200
  fear/greed               79.0


## Determinism

The same seed and universe give the same market, bit for bit, on every
platform the library ships for. CI builds all five of those targets, runs one
fixed simulation inside each wheel, and fails if any two digests disagree.
The WebAssembly build gets the same treatment on its own fixed simulation,
but by hand rather than in CI, because it needs a separate toolchain. Here is
the local half.

In [6]:
def run(seed, universe_seed=111, days=10):
    u = tf.Universe.random(20, seed=universe_seed)
    e = tf.Engine(universe=u, seed=seed)
    e.run_days(days)
    r = e.prices()
    return struct.unpack(f"<{len(r) // 8}d", r)

a = run(7)
b = run(7)
c = run(8)

print("same seed, run twice     :", "identical" if a == b else "DIFFERENT")
print("different seed           :", "identical" if a == c else "different, as expected")
print()
print(f"  seed 7 first price: {a[0]:.10f}")
print(f"  seed 8 first price: {c[0]:.10f}")

same seed, run twice     : identical
different seed           : different, as expected

  seed 7 first price: 193.7100000000
  seed 8 first price: 189.4000000000


## Provenance

Every run carries fingerprints identifying exactly what produced it. Cite
these alongside any result.

In [7]:
print("model preset     :", engine.model_fingerprint)
print("universe (20,111):", tf.universe_util.fingerprint_of(universe)[:32], "...")
print()
print("A different preset is a different market, and says so:")
for name in ("pt-v1", "pt-v3", "pt-v4"):
    print(f"  {name}  ->  {tf.ModelParams.from_preset(name).fingerprint}")

model preset     : pt-v16
universe (20,111): 804aa49d57a50b30a31d7467c6e93153 ...

A different preset is a different market, and says so:
  pt-v1  ->  pt-v1
  pt-v3  ->  pt-v3
  pt-v4  ->  pt-v4


## Next

- **[2 · Evaluating a strategy](02-evaluating-a-strategy.ipynb)**: is my
  strategy any good, and how would I know?
- **[3 · Why did the price move](03-why-did-the-price-move.ipynb)**: the
  nine factors that sum to every move away from fair value.
- **[4 · How realistic is this](04-how-realistic-is-this.ipynb)**: what
  this is certified to reproduce, and where it fails.

Full documentation: <https://simoncoombes.github.io/tradefloor/>